# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

*Probeprüfung 2 (Übungsversion)*

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (fitness.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (fitness.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

Die Daten stammen von der **fiktiven** Schweizer Fitnesskette «FitNetz» mit sieben Studios. Mitglieder buchen einzelne Kurstermine (z.B. Indoor Cycling am Montag, 18:00 Uhr im Studio Oerlikon).

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `fitness.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `studios`, `trainers`, `courses`, `sessions`, `members`, `bookings`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `fitness_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

### Datenbankverbindung zu PostgreSQL

In [ ]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

### Daten nach PostgreSQL laden

In [ ]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['studios', 'trainers', 'courses', 'sessions', 'members', 'bookings']

sqlite_engine = create_engine('sqlite:///fitness.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `fitness.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///fitness.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `fitness.db` werden nicht gewertet.*

### A1 (4 Punkte)

Mitglieder (`members`) und Kurstermine (`sessions`) sind in `fitness.db` über die Tabelle `bookings` miteinander verbunden.

Ein Entwickler schlägt vor: *«Die Tabelle `bookings` ist überflüssig. Wir ergänzen in `sessions` einfach eine Spalte `member_ids`, in der alle Teilnehmenden stehen, z.B. `'MB0012,MB0087,MB0311'`.»*

a) Welche Kardinalität hat die Beziehung zwischen `members` und `sessions`? Begründen Sie mit einem Beispiel aus dem Studiobetrieb.

b) Nennen und begründen Sie **zwei** konkrete Nachteile des Vorschlags. Beziehen Sie sich dabei auf Spalten bzw. Abfragen von `fitness.db`.

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Das Datenteam prüft die Spalte `email` der Tabelle `members` und findet unter anderem folgende Einträge:

```
noah.fischer119@gmail.com
NULL
mia.moser54.gmail.com
leon.zuercher141(at)bluewin.ch
NULL
nico.moser299 gmx.ch
```

Beschreiben Sie die **zwei unterschiedlichen** Datenprobleme und ordnen Sie jedem Problem eine passende Data-Quality-Eigenschaft aus den Vorlesungsfolien zu (accurate, consistent, complete, reliable). Nennen Sie eine konkrete Folge für FitNetz.

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

In `fitness.db` hat kein Kurstermin mehr Buchungen als seine `capacity`. Angenommen, für einen neuen Termin mit `capacity` = 16 werden 18 Buchungen erfasst: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `fitness.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Umsatz (Summe von `paid_fee`) pro Studio, für **alle** Studios in der Datenbank.

```sql
SELECT st.name, SUM(b.paid_fee) AS revenue
FROM bookings b
JOIN sessions s ON b.sessionid = s.sessionid
JOIN studios st ON s.studioid = st.studioid
GROUP BY st.name;
```

Die Tabelle `studios` enthält **7 Studios**. Tatsächliches Ergebnis der SQL-Abfrage: nur **6 Zeilen**, Umsatz zwischen CHF 7'000 und CHF 7'800.

Die Geschäftsleitung fragt: *«Unser neues Studio in Zug fehlt. Es soll in der Liste erscheinen, und zwar mit einem Umsatz von 0 und nicht mit einem leeren Feld.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 7 Studios** erscheinen und ein Studio ohne Buchungen mit `0` ausgewiesen wird. Runden Sie den Umsatz mit `ROUND(CAST(... AS NUMERIC), 2)` und sortieren Sie absteigend nach `revenue`. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B1

### B2 (8 Punkte)

**Ziel:** Liste aller Trainer:innen (`trainerid`, `firstname`, `lastname`), die noch **keinen** Kurstermin geleitet haben (für die Einsatzplanung).

```sql
SELECT t.trainerid, t.firstname, t.lastname
FROM trainers t
WHERE t.trainerid NOT IN (SELECT s.trainerid FROM sessions s);
```

Tatsächliches Ergebnis: **0 Zeilen**, obwohl die Personalabteilung weiss, dass mindestens zwei kürzlich eingestellte Trainer:innen noch keinen Termin geleitet haben.

**Aufgabe:** Finden Sie die Ursache, schreiben Sie die korrigierte SQL-Abfrage, die alle Trainer:innen ohne Kurstermin ausgibt, und begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

*Tipp: Prüfen Sie die Werte in `sessions.trainerid`.*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B2

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `fitness.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Laut Geschäftsbedingungen sind nur Mitglieder ab 16 Jahren zugelassen; Geburtsdaten vor dem 01.01.1920 gelten als unplausibel. Schreiben Sie eine SQL-Abfrage, die alle Mitglieder (`memberid`, `firstname`, `lastname`, `birthdate`) mit einem **ungültigen** Geburtsdatum ausgibt, d.h. fehlend, vor dem `1920-01-01` oder nach dem `2010-12-31`. Sortieren Sie nach `memberid`. Begründen Sie Ihre Filterbedingung.

*Hinweis: `birthdate` ist als Text im Format `JJJJ-MM-TT` gespeichert. Vergleiche wie `birthdate < '1920-01-01'` funktionieren in beiden Datenbanken.*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C1

### C2 (4 Punkte)

Wie viele Buchungen gibt es pro Kurskategorie (`category`), und wie hoch ist der durchschnittlich bezahlte Preis (`paid_fee`)? Geben Sie `category`, die Anzahl Buchungen (`booking_count`) und den Durchschnittspreis (`avg_fee`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_fee`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C2

### C3 (4 Punkte)

FitNetz vergibt eine Treueprämie an alle Mitglieder mit **mindestens 25 Buchungen**. Geben Sie `memberid`, `firstname`, `lastname` und die Anzahl Buchungen (`booking_count`) aus, absteigend sortiert nach `booking_count`, bei Gleichstand aufsteigend nach `memberid`. Begründen Sie, warum die Bedingung «mindestens 25 Buchungen» nicht in der `WHERE`-Klausel stehen kann.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C3

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')